# Piblo persona: distil the full Talk prompt into Qwen3-1.7B (LoRA)

Core already runs Qwen3-1.7B locally (llama.cpp). Today Piblo's personality lives in a long system prompt
(`host/mazhost/prompts.py`): slow first words, and it still drifts. This notebook:
1. A bigger open model (Qwen3-8B, 4-bit) answers ~4,000 varied user turns *with* the full prompt (teacher).
2. Answers are filtered by Piblo's own rules (25 words max, banned phrases, uses the facts given).
3. Qwen3-1.7B gets a LoRA trained on them with a one-line prompt (student).
4. Held-out check: student (short prompt) vs base (full prompt) on the same rules.
5. Exports a q4_k_m GGUF for Core's llama.cpp.

Kaggle: GPU T4, Internet On. Add `host/mazhost/prompts.py` as a dataset (any name). Built on Unsloth (Apache-2.0).

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps trl

In [ ]:
import glob, json, random, re, ast
random.seed(7)
found = glob.glob("/kaggle/input/**/prompts.py", recursive=True)
assert found, "add host/mazhost/prompts.py as a Kaggle dataset"
tree = ast.parse(open(found[0], encoding="utf-8").read())
SYSTEM_PROMPT = next(n.value.value for n in tree.body if isinstance(n, ast.Assign) and getattr(n.targets[0], "id", "") == "SYSTEM_PROMPT")
SHORT_PROMPT = "You are Piblo, the owner's warm, quick pocket voice sidekick. Spoken aloud: short, plain, no markdown."
print(len(SYSTEM_PROMPT), "chars of persona to distil")

In [ ]:
# Varied user turns. Facts arrive the way Core sends them: a <turn_facts> block.
TOPICS = ["what time is it", "what's the date", "weather today", "tell me a joke", "quick maths", "how do I boil an egg",
          "what should I work on next", "remind me later", "I'm so tired", "everything's broken", "I did it, it works",
          "guess what", "be a night owl", "calm down", "who are you", "what model are you", "and after that?",
          "explain something simply", "give me advice", "motivate me", "plan my evening", "fun fact", "spell a word",
          "convert units", "recommend a film", "I can't focus", "say something kind", "how long until Friday"]
WEATHER = ["light rain, 9C", "sunny, 14C", "cloudy, 11C", "windy, 7C", "clear, 3C"]
def facts():
    h, m = random.randint(6, 23), random.choice([0, 5, 15, 20, 30, 45, 50])
    day = random.choice(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    return f"<turn_facts>Time: {h:02d}:{m:02d}. Day: {day}. Location: UK. Weather: {random.choice(WEATHER)}.</turn_facts>"

In [ ]:
from unsloth import FastLanguageModel
teacher, ttok = FastLanguageModel.from_pretrained("unsloth/Qwen3-8B-unsloth-bnb-4bit", max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(teacher)
ttok.padding_side = "left"

def chat(model, tok, system, users, max_new=90, temp=0.8):
    texts = [tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": u}],
                                     tokenize=False, add_generation_prompt=True, enable_thinking=False) for u in users]
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new, temperature=temp, do_sample=True, top_p=0.9)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

# 1) many natural phrasings per topic
users = []
for t in TOPICS:
    for _ in range(6):
        lines = chat(teacher, ttok, "Write 25 different short things a person might say out loud to a voice assistant. "
                     "One per line, no numbering, casual UK English.", [f"Theme: {t}"], max_new=600, temp=1.0)[0]
        users += [l.strip(" -*\t") for l in lines.splitlines() if 3 < len(l.strip()) < 120]
users = list(dict.fromkeys(users))
random.shuffle(users)
print(len(users), "user turns")

In [ ]:
# 2) teacher answers with the full persona; 3) keep only answers that obey Piblo's rules
BANNED = re.compile(r"(?i)as an ai|i don't have access|i can't|i am unable|i'm unable|language model|qwen|alibaba|\*\*|#")
def ok(ans):
    return bool(ans) and len(ans.split()) <= 25 and not BANNED.search(ans) and "turn_facts" not in ans

rows = []
for i in range(0, len(users), 16):
    batch = [f"{facts()}\n{u}" for u in users[i:i + 16]]
    for u, a in zip(batch, chat(teacher, ttok, SYSTEM_PROMPT, batch)):
        if ok(a):
            rows.append({"user": u, "answer": a})
print(len(rows), "kept of", len(users))
random.shuffle(rows)
test, train = rows[:150], rows[150:]
json.dump(rows, open("/kaggle/working/piblo_persona.json", "w"), indent=1)
del teacher
import torch, gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4) LoRA on Qwen3-1.7B with the short prompt
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
model, tok = FastLanguageModel.from_pretrained("unsloth/Qwen3-1.7B", max_seq_length=1024, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
def fmt(r):
    return {"text": tok.apply_chat_template([{"role": "system", "content": SHORT_PROMPT}, {"role": "user", "content": r["user"]},
            {"role": "assistant", "content": r["answer"]}], tokenize=False, enable_thinking=False)}
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=Dataset.from_list(train).map(fmt),
    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=8, gradient_accumulation_steps=2,
                   num_train_epochs=2, learning_rate=2e-4, logging_steps=20, output_dir="out", report_to="none"))
trainer.train()

In [ ]:
# 5) Held-out check on Piblo's rules: student + short prompt vs base + full prompt
FastLanguageModel.for_inference(model)
tok.padding_side = "left"
def score(answers, rows):
    timed = [(a, r) for a, r in zip(answers, rows) if "time" in r["user"].lower()]
    t_ok = sum(re.search(r"Time: (\d\d):(\d\d)", r["user"]).group(1).lstrip("0") in a for a, r in timed)
    return {"rule_pass": round(sum(ok(a) for a in answers) / len(rows), 3), "time_correct": f"{t_ok}/{len(timed)}",
            "avg_words": round(sum(len(a.split()) for a in answers) / len(answers), 1)}
users_t = [r["user"] for r in test]
student = sum((chat(model, tok, SHORT_PROMPT, users_t[i:i + 16], temp=0.3) for i in range(0, len(users_t), 16)), [])
with model.disable_adapter():
    base = sum((chat(model, tok, SYSTEM_PROMPT, users_t[i:i + 16], temp=0.3) for i in range(0, len(users_t), 16)), [])
report = {"student_short_prompt": score(student, test), "base_full_prompt": score(base, test),
          "samples": [{"user": u, "student": s, "base": b} for u, s, b in list(zip(users_t, student, base))[:20]]}
json.dump(report, open("/kaggle/working/eval.json", "w"), indent=1)
print(json.dumps({k: report[k] for k in ("student_short_prompt", "base_full_prompt")}, indent=1))

In [ ]:
# 6) Export for Core's llama.cpp (ship only if the student beat the base above)
model.save_pretrained_gguf("/kaggle/working/piblo-qwen3-1.7b", tok, quantization_method="q4_k_m")
print(glob.glob("/kaggle/working/**/*.gguf", recursive=True))